In [1]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import numpy as np
import pandas as pd
import ast
import glob
import pickle
import dask
import os
import itertools

from tqdm.notebook import tqdm

#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score


from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed

import dask
import dask.dataframe as dd
from dask.distributed import Client
#client = Client(n_workers=20, memory_limit="10GB", interface='lo')
from concurrent.futures import ThreadPoolExecutor

import dask_ml.cluster as dask_cluster

from pprint import pprint
import os

### Load and Compile Old Backtests

In [2]:
DATA_FOLDER = "../../data/output/backtest_state_forests_windowsize=2"
file_paths = [os.path.join(DATA_FOLDER, file) for file in os.listdir(DATA_FOLDER) if file.endswith(".csv")]

def read_csv(file_path):
    return pd.read_csv(file_path)

with tqdm(total=len(file_paths), desc="Reading CSV files") as pbar:
    # Define the function for reading CSV files
    # Read the CSV files in parallel
    TLGRF_dfs = Parallel(n_jobs=-1)(delayed(read_csv)(file_path) for file_path in file_paths)
    pbar.update(len(file_paths))  # Manually update the progress bar


In [3]:
TLGRF_df = pd.concat(TLGRF_dfs).sort_values(by=["fips", "days_from_start"])
TLGRF_df["datetime"] = pd.to_datetime(TLGRF_df["datetime"])
TLGRF_df = TLGRF_df.rename(columns={"log_rolled_cases.x": "log_rolled_cases", "predicted.grf.future.last":"TLGRF_predicted_log_rolled_cases", "t0.hat":"intercept_TLGRF", "tau.hat":"r_TLGRF"})
TLGRF_df = TLGRF_df[["fips","county","state","days_from_start","datetime", "intercept_TLGRF","r_TLGRF","log_rolled_cases", "TLGRF_predicted_log_rolled_cases"]]
TLGRF_df["shifted_log_rolled_cases"] = TLGRF_df.groupby("fips")["log_rolled_cases"].shift(-7)
TLGRF_df.to_csv("benchmark_TLGRF_dataset.csv", index=False)
display(TLGRF_df)

,fips,county,state,days_from_start,datetime,intercept_TLGRF,r_TLGRF,log_rolled_cases,TLGRF_predicted_log_rolled_cases,shifted_log_rolled_cases
0,1001,Autauga,Alabama,87,2020-04-17,-26.766326,0.028066,3.030824,3.227285,3.122994
0,1001,Autauga,Alabama,88,2020-04-18,-313.778047,0.007752,3.030824,3.085088,3.160035
0,1001,Autauga,Alabama,89,2020-04-19,-60.990643,0.021119,3.044522,3.192355,3.183989
0,1001,Autauga,Alabama,90,2020-04-20,-54.020959,0.022045,3.064725,3.219041,3.213145
0,1001,Autauga,Alabama,91,2020-04-21,-216.383634,0.010171,3.064725,3.135925,3.241476
...,...,...,...,...,...,...,...,...,...,...
2222,99999,New York City,New York,1153,2023-03-19,1456.032249,-0.030986,9.544831,9.327926,NaN
2212,99999,New York City,New York,1154,2023-03-20,1389.214236,-0.039535,9.496840,9.220095,NaN
2194,99999,New York City,New York,1155,2023-03-21,1323.021840,-0.054503,9.430176,9.048656,NaN
2190,99999,New York City,New York,1156,2023-03-22,1531.624962,-0.024711,9.405484,9.232510,NaN


In [4]:
TLGRF_df.head(8)

,fips,county,state,days_from_start,datetime,intercept_TLGRF,r_TLGRF,log_rolled_cases,TLGRF_predicted_log_rolled_cases,shifted_log_rolled_cases
0,1001,Autauga,Alabama,87,2020-04-17,-26.766326,0.028066,3.030824,3.227285,3.122994
0,1001,Autauga,Alabama,88,2020-04-18,-313.778047,0.007752,3.030824,3.085088,3.160035
0,1001,Autauga,Alabama,89,2020-04-19,-60.990643,0.021119,3.044522,3.192355,3.183989
0,1001,Autauga,Alabama,90,2020-04-20,-54.020959,0.022045,3.064725,3.219041,3.213145
0,1001,Autauga,Alabama,91,2020-04-21,-216.383634,0.010171,3.064725,3.135925,3.241476
0,1001,Autauga,Alabama,92,2020-04-22,-130.688649,0.014341,3.071370,3.171760,3.274446
0,1001,Autauga,Alabama,93,2020-04-23,-87.995913,0.017458,3.084528,3.206731,3.311585
0,1001,Autauga,Alabama,94,2020-04-24,-0.872497,0.034882,3.122994,3.367166,3.306363


In [5]:
TLGRF_df.shape

(2636735, 10)

In [7]:
TLGRF_df["TLGRF_predicted_log_rolled_cases"].isna().sum()

0

In [8]:
len(TLGRF_df["days_from_start"].unique()) * len(TLGRF_df["fips"].unique())

3464910